In [ ]:
import pickle
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

In [149]:
df = pd.read_csv("../data/data_transformed.csv", keep_default_na=False)

In [150]:
X = df['transformed_message']

In [151]:
y = df['label'].values

In [152]:
X_train,X_test, y_train , y_test = train_test_split(X, y, test_size=0.2, random_state=2,stratify=y)

In [153]:
classifiers = {
    'DT' : DecisionTreeClassifier(max_depth=5),
    'RF' : RandomForestClassifier(n_estimators=50, random_state=2),
    'GNB' : GaussianNB(),
    'BNB' : BernoulliNB(),
    'MNB' : MultinomialNB()
}

In [154]:
def to_dense(X):
    # Needed for Gaussian NB
    return X.toarray()

In [155]:
cv_pipelines = {}
tfidf_pipelines = {}

In [156]:
for name, classifier in classifiers.items():

    # CountVectorizer Pipeline
    cv_steps = [
        ('vectorizer', CountVectorizer())
    ]
    
    if name == 'GNB':
        cv_steps.append(
            ('to_dense', FunctionTransformer(to_dense))
        )

    cv_steps.append(
        ('classifier', classifier)
    )

    cv_pipelines[name] = Pipeline(cv_steps)

    # Tf-Idf Pipeline
    tfidf_steps = [
        ('vectorizer', TfidfVectorizer(max_features=3000))
    ]

    if name == 'GNB':
        tfidf_steps.append(
            ('to_dense', FunctionTransformer(to_dense))
        )

    tfidf_steps.append(
        ('classifier', classifier)
    )

    tfidf_pipelines[name] = Pipeline(tfidf_steps)

In [157]:
def evaluate_pipelines(pipelines, X_train, X_test, y_train, y_test):

    results = []

    for name, pipeline in pipelines.items():

        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)

    
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test,y_pred)
        recall = recall_score(y_test,y_pred)
        f1 = f1_score(y_test,y_pred)

        results.append({
            'Algorithm': name,
            'Accuracy': accuracy,
            'Precision': precision,
            'Recall': recall,
            'F1-Score': f1
        })

    return pd.DataFrame(results)

In [158]:
cv_performance_df = evaluate_pipelines(
    cv_pipelines,
    X_train,
    X_test,
    y_train,
    y_test
)

In [159]:
cv_performance_df.sort_values(by='Precision', ascending=False)

,Algorithm,Accuracy,Precision,Recall,F1-Score
1,RF,0.977756,1.000000,0.824427,0.903766
3,BNB,0.971954,1.000000,0.778626,0.875536
4,MNB,0.987427,0.960938,0.938931,0.949807
0,DT,0.930368,0.953846,0.473282,0.632653
2,GNB,0.876209,0.506329,0.916031,0.652174


In [160]:
tfidf_performance_df = evaluate_pipelines(
    tfidf_pipelines,
    X_train,
    X_test,
    y_train,
    y_test
)

In [161]:
tfidf_performance_df.sort_values(by='Precision', ascending=False)

,Algorithm,Accuracy,Precision,Recall,F1-Score
1,RF,0.980658,1.000000,0.847328,0.917355
4,MNB,0.978723,1.000000,0.832061,0.908333
3,BNB,0.988395,0.991736,0.916031,0.952381
0,DT,0.939072,0.869565,0.610687,0.717489
2,GNB,0.849130,0.451362,0.885496,0.597938


In [167]:
model = tfidf_pipelines['BNB']

In [174]:
with open('../models/model.pkl', 'wb') as f:
    pickle.dump(model, f)